# Mistral FP16 → AWQ: measured GPU workflow

Run **one cell at a time, top to bottom**. Do not use “Run all”. Keep the same A100/L4
runtime for the cloud stages. Existing completed results and the AWQ checkpoint are
verified and reused; the setup cell does not recreate an existing GPU environment.

Order: **1 setup → 2 isolated FP16 baseline/review → 3 quantization + immediate paired
quality evaluation → 4 FP16 then AWQ serving sweeps → 5 laptop handoff → 6 measured report**.
Stop at every error and share the complete printed log. No automatic package downgrades,
precision/model fallbacks, or invented results. Expect nominal 24 GB+ VRAM and 32 GB+
host RAM. Select A100/L4 under **Runtime → Change runtime type**.

**Resuming your existing session:** stop any running cell/server in the old notebook first.
Use this updated notebook with the same runtime if possible. Do not delete `/content`,
reset the runtime, or rerun the old setup cell. If Colab gives this notebook a different
runtime, restore saved artifacts before continuing; ephemeral files do not move with a link.
The GPU subprocess environment explicitly uses `MPLBACKEND=Agg`; no inline Matplotlib
backend needs to be installed in the separate GPU environment.

## Step 1 — Configure and verify the GPU environment

The repository URL and source revision are already filled in. An existing GPU environment
is checked against the pinned direct dependencies without changing installed versions.
A missing environment is installed from the committed lockfile. Errors stop the cell.
Existing model artifacts and results are preserved.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/rithwik0908/llm-inference-benchmark.git"
REPO_COMMIT = "0ef7a6f148ceae7af27b91fbf1bbd427b74ecb53"
ROOT = Path("/content/llm-inference-benchmark")
GPU_ENV = Path("/content/benchmark-venv")
GPU_PY = GPU_ENV / "bin/python"

# Colab's inline plotting backend is unavailable inside the GPU virtualenv.
# Set this explicitly for every subprocess, including vLLM CLI imports.
PROC_ENV = os.environ.copy()
PROC_ENV["MPLBACKEND"] = "Agg"

if not ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True, env=PROC_ENV)
subprocess.run(["git", "-C", str(ROOT), "fetch", "origin"], check=True, env=PROC_ENV)
subprocess.run(["git", "-C", str(ROOT), "checkout", REPO_COMMIT], check=True, env=PROC_ENV)
os.chdir(ROOT)

if not GPU_PY.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.9.26"], check=True, env=PROC_ENV)
    subprocess.run(
        ["uv", "venv", str(GPU_ENV), "--python", "3.11", "--seed"], check=True, env=PROC_ENV
    )
    subprocess.run(
        ["uv", "pip", "sync", "--python", str(GPU_PY), "requirements/gpu.lock.txt"],
        cwd=ROOT,
        check=True,
        env=PROC_ENV,
    )
    subprocess.run(
        ["uv", "pip", "install", "--python", str(GPU_PY), "--no-deps", "-e", "."],
        cwd=ROOT,
        check=True,
        env=PROC_ENV,
    )
else:
    print("Reusing existing GPU environment; no packages will be changed.")

# Check direct runtime pins, allowing a local CUDA wheel version suffix.
version_check = """
import importlib.metadata as m
import json, pathlib, tomllib
from packaging.version import Version
p = tomllib.loads(pathlib.Path('pyproject.toml').read_text())['project']
versions = {}
for spec in p['dependencies'] + p['optional-dependencies']['gpu']:
    name, expected = spec.split('==')
    actual = m.version(name)
    if Version(actual).base_version != expected:
        raise RuntimeError(
            f'Pinned version mismatch: {name}: {actual} != {expected}. STOP; do not auto-fix.'
        )
    versions[name] = actual
print(json.dumps(versions, indent=2))
"""
subprocess.run([str(GPU_PY), "-c", version_check], cwd=ROOT, env=PROC_ENV, check=True)
print("Environment verified. GPU subprocess plotting backend:", PROC_ENV["MPLBACKEND"])

### Runtime helpers — execute this cell once

The runner prints command output and saves logs. Startup failures show the underlying
traceback. Quantization has a one-hour hard stop; quality commands have 15-minute limits;
each full serving sweep has a four-hour hard stop with an elapsed-time heartbeat every
five minutes. A heartbeat is progress information, not a benchmark result. Timeouts and
interrupts terminate the process group; inspect logs instead of retrying blindly.

In [ ]:
import hashlib
import signal
import threading
import time
import urllib.error
import urllib.request
from datetime import UTC, datetime

server = None
server_log = None
BASELINE_OK = False
QUALITY_OK = False
FP16_SWEEP_OK = False
AWQ_SWEEP_OK = False
EDGE_OK = False


def load_result(name):
    return json.loads((ROOT / "results" / name).read_text())


def content_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def terminate_group(process):
    try:
        os.killpg(process.pid, signal.SIGTERM)
    except ProcessLookupError:
        pass
    try:
        process.wait(timeout=30)
    except subprocess.TimeoutExpired:
        try:
            os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        process.wait(timeout=15)


def run(*args, timeout_s=1800):
    logs = ROOT / "results"
    logs.mkdir(exist_ok=True)
    stamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
    log_path = logs / f"{Path(args[0]).stem}-{stamp}.log"
    command = [str(GPU_PY), *args]
    print("Running:", command, "\nLog:", log_path, flush=True)
    env = {**PROC_ENV, "MPLBACKEND": "Agg", "PYTHONUNBUFFERED": "1"}
    process = subprocess.Popen(
        command,
        cwd=ROOT,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        errors="replace",
        start_new_session=True,
    )

    def relay():
        with log_path.open("w") as log:
            for line in process.stdout:
                log.write(line)
                log.flush()
                print(line, end="", flush=True)

    reader = threading.Thread(target=relay, daemon=True)
    reader.start()
    start = time.monotonic()
    last_heartbeat = start
    try:
        while process.poll() is None:
            elapsed = time.monotonic() - start
            if elapsed > timeout_s:
                raise TimeoutError(
                    f"STOP: exceeded {timeout_s / 60:.0f} minutes; inspect {log_path}"
                )
            if time.monotonic() - last_heartbeat >= 300:
                print(f"Still running: {elapsed / 60:.1f} minutes. Log: {log_path}", flush=True)
                last_heartbeat = time.monotonic()
            time.sleep(0.2)
    except BaseException:
        terminate_group(process)
        raise
    finally:
        reader.join(timeout=10)
        process.stdout.close()
    if process.returncode:
        print(log_path.read_text(errors="replace")[-20000:])
        raise subprocess.CalledProcessError(process.returncode, command)
    print(f"Completed in {(time.monotonic() - start) / 60:.1f} minutes", flush=True)


def stop_server():
    global server, server_log
    if server is not None:
        terminate_group(server)
        server = None
    if server_log is not None:
        server_log.close()
        server_log = None


def start_server(variant):
    global server, server_log
    stop_server()
    path = ROOT / "results" / f"server_{variant}.log"
    # Preserve previous startup diagnostics instead of silently replacing them.
    if path.exists():
        stamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
        path.rename(path.with_name(f"server_{variant}-{stamp}.log"))
    server_log = path.open("w")
    server = subprocess.Popen(
        [str(GPU_PY), "-m", "inference_bench.server", variant],
        cwd=ROOT,
        env={**PROC_ENV, "MPLBACKEND": "Agg", "PYTHONUNBUFFERED": "1"},
        stdout=server_log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    deadline = time.monotonic() + 900
    try:
        while time.monotonic() < deadline:
            if server.poll() is not None:
                raise RuntimeError("Server exited:\n" + path.read_text(errors="replace")[-20000:])
            try:
                with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=5) as response:
                    if response.status == 200:
                        return
            except (urllib.error.URLError, TimeoutError):
                pass
            time.sleep(2)
        raise TimeoutError(
            "Server startup exceeded 15 minutes:\n" + path.read_text(errors="replace")[-20000:]
        )
    except BaseException:
        stop_server()
        raise


run("scripts/preflight.py", "--min-vram-gib", "22", "--output", "results/preflight_colab.json")
preflight = load_result("preflight_colab.json")
cfg = json.loads((ROOT / "configs/experiment.json").read_text())
expected_identity = {"model": cfg["model"], "revision": cfg["revision"]}
print("GPU:", preflight["gpu"])

## Step 2 — Isolated FP16 baseline, then pause for review

Only the baseline runs here; no vLLM server, quantization, or quality evaluation starts.
Existing results are reused only if their model, workload, and physical GPU match.
The memory guard uses **decimal GB**, approximately 14–16 GB for this short-context
Mistral workload. Throughput must be finite and at least 1 token/s; that is a broad
near-zero guard, not proof of hardware efficiency. Review the actual printed values.
Any failed guard stops the workflow. No offloading or automatic “fix” is attempted.

**Pause after this cell and share the printed baseline summary before Step 3.**

In [ ]:
import math

stop_server()
BASELINE_OK = False
for script, artifact in (
    ("prepare_workload.py", "workload.json"),
    ("prepare_eval.py", "eval_set.json"),
):
    if not (ROOT / "results" / artifact).exists():
        run(f"scripts/{script}")
workload = load_result("workload.json")
eval_set = load_result("eval_set.json")
assert len(workload["prompts"]) == 512, "STOP: expected the frozen 512-prompt workload"
assert len(eval_set["questions"]) == 50, "STOP: expected the frozen 50-question set"
baseline_path = ROOT / "results/baseline_fp16.json"
if not baseline_path.exists():
    run("scripts/baseline_fp16.py")
else:
    print("Reusing existing FP16 baseline; no model reload.")
baseline = load_result("baseline_fp16.json")
assert baseline["identity"] == expected_identity, "STOP: baseline model mismatch"
assert baseline["workload_id"] == workload["workload_id"], "STOP: baseline workload mismatch"
assert baseline["gpu"]["uuid"] == preflight["gpu"]["uuid"], "STOP: baseline used a different GPU"
print("GPU:", baseline["gpu"]["name"])
print("Summary:", json.dumps(baseline["summary"], indent=2))
print("Peak allocated VRAM (decimal GB):", baseline["peak_allocated_bytes"] / 1e9)
print("Peak reserved VRAM (decimal GB):", baseline["peak_reserved_bytes"] / 1e9)

rate = baseline["summary"]["output_tokens_per_s_p50"]
assert math.isfinite(rate) and rate >= 1, (
    "STOP: nonfinite or near-zero throughput; share exact output"
)
assert 14e9 <= baseline["peak_allocated_bytes"] <= 16e9, (
    "STOP: peak allocated memory is outside the requested 14–16 GB sanity range; share exact output"
)
assert all(r["output_tokens"] == 128 for r in baseline["rows"]), "STOP: wrong token budget"
BASELINE_OK = True
BASELINE_HASH = content_hash(baseline_path)
print("Step 2 saved and basic checks passed. PAUSE and report the actual values before Step 3.")

## Step 3 — Quantization and quality evaluation together

Run this cell only after reviewing Step 2. It reuses a complete existing AWQ checkpoint
and verifies its hashes; an incomplete checkpoint is never overwritten. Otherwise it
calibrates AWQ with a one-hour hard stop. **AWQ quality is evaluated immediately next**,
then FP16 quality, on the same frozen 50 questions. There is no load test in this cell.
Completed same-checkpoint/same-GPU quality files can also be reused.

The reported delta is AWQ minus FP16 in **percentage points**. A drop exceeding 3 pp is
printed prominently and stops progression to serving for review. This is a real finding,
not an environment bug to hide. One answer is 2 pp; this is not full MMLU.
**Report the two accuracies and delta before Step 4.**

In [ ]:
assert BASELINE_OK, "Run and review Step 2 first"
assert content_hash(ROOT / "results/baseline_fp16.json") == BASELINE_HASH
QUALITY_OK = False
stop_server()
checkpoint = ROOT / "models/mistral-7b-awq"
manifest_path = checkpoint / "benchmark_manifest.json"
if manifest_path.exists() and (ROOT / "results/quantization.json").exists():
    print("Reusing completed quantization; verifying checkpoint content hashes.")
else:
    if checkpoint.exists() and any(checkpoint.iterdir()):
        raise RuntimeError(
            "STOP: incomplete/nonempty checkpoint. Preserve it and report; do not overwrite."
        )
    run("scripts/quantize_awq.py", timeout_s=3600)
run(
    "-c",
    "from inference_bench.server import checkpoint_manifest; "
    "print('Verified checkpoint:', checkpoint_manifest('models/mistral-7b-awq')['checkpoint_id'])",
)
quantization = load_result("quantization.json")
manifest = json.loads(manifest_path.read_text())
assert quantization["checkpoint_id"] == manifest["checkpoint_id"], (
    "STOP: quantization record mismatch"
)
print("Measured quantization seconds:", quantization["quantization_s"])
for variant in ("awq", "fp16"):
    quality_path = ROOT / "results" / f"quality_{variant}.json"
    expected_checkpoint = (
        manifest["checkpoint_id"]
        if variant == "awq"
        else hashlib.sha256(
            json.dumps(expected_identity, sort_keys=True, separators=(",", ":")).encode()
        ).hexdigest()
    )
    if quality_path.exists():
        saved = json.loads(quality_path.read_text())
        assert saved["identity"] == expected_identity
        assert saved["eval_set_id"] == eval_set["eval_set_id"]
        assert saved["server"]["checkpoint_id"] == expected_checkpoint
        assert saved["server"]["gpu"]["uuid"] == baseline["gpu"]["uuid"]
        assert len(saved["rows"]) == 50
        print(f"Reusing completed {variant.upper()} quality result.")
    else:
        try:
            start_server(variant)
            run(
                "scripts/evaluate_quality.py",
                "run",
                "--server-manifest",
                f"results/server_{variant}.json",
                timeout_s=900,
            )
        finally:
            stop_server()
run("scripts/evaluate_quality.py", "compare")
quality = load_result("quality_delta.json")
print("\nQUALITY RESULTS")
for key in (
    "questions",
    "fp16_accuracy_pct",
    "awq_accuracy_pct",
    "accuracy_delta_pp_awq_minus_fp16",
    "paired_bootstrap_95ci_pp",
    "fp16_invalid",
    "awq_invalid",
):
    print(key, quality[key])
if quality["accuracy_delta_pp_awq_minus_fp16"] < -3:
    raise RuntimeError(
        "QUALITY FINDING: accuracy dropped more than 3 pp. STOP and report before serving."
    )
QUALITY_OK = True
QUALITY_HASH = content_hash(ROOT / "results/quality_delta.json")
print("Step 3 complete. PAUSE and report quality results before starting Step 4.")

## Step 4 — Serving sweeps only

Run FP16 first, then AWQ. Each uses **512 frozen prompts**, concurrency **1 / 4 / 16 / 32**,
and **three repetitions** with fixed 256/128 token budgets. Quality is not rerun here.
A low-concurrency repetition can take tens of minutes; a five-minute elapsed-time
heartbeat distinguishes a running command from a silent notebook. Per-request HTTP
requests have their own timeouts, and each entire variant has a four-hour hard stop.

CUDA/driver/dependency failures stop immediately and print logs. Do not downgrade,
reinstall, or start the other variant after a failure. Existing invalid/partial sweep
files are preserved for diagnosis, never relabeled successful. Report after each variant.

**Measurement protocol 2:** requests ask vLLM for delta generated-token IDs. TTFT is
first generated-ID arrival, including tokens that render no text. First-visible-text
arrival is separately recorded and may be absent. Success requires exactly 128 streamed
IDs as well as matching usage, finish reason, and `[DONE]`. Extra ID transport is part
of both variants' measurements. Generated-token throughput includes invisible special
tokens; the report counts empty-text completions explicitly. Old text-only TTFT sweeps
must be preserved and cannot be mixed with the corrected runs. Model weights, prompts,
precision, token budgets, and the separate quality evaluation are unchanged.


In [ ]:
def sweep(variant):
    assert QUALITY_OK, "Complete and review Step 3 first"
    assert content_hash(ROOT / "results/quality_delta.json") == QUALITY_HASH
    path = ROOT / "results" / f"serving_{variant}.json"
    if path.exists():
        result = json.loads(path.read_text())
        if not result.get("valid"):
            raise RuntimeError(
                f"STOP: partial/invalid sweep at {path}; preserve it and report before retrying"
            )
        print("Reusing completed sweep:", path)
    else:
        try:
            start_server(variant)
            run(
                "scripts/benchmark_serving.py",
                "--server-manifest",
                f"results/server_{variant}.json",
                timeout_s=4 * 3600,
            )
        finally:
            stop_server()
        result = json.loads(path.read_text())
    assert result.get("measurement_protocol", {}).get("version") == 2, (
        "STOP: old TTFT protocol. Preserve this artifact and rerun under token-ID timing."
    )
    assert result["valid"] and result["variant"] == variant
    assert result["identity"] == expected_identity
    assert result["workload_id"] == workload["workload_id"]
    assert result["server"]["gpu"]["uuid"] == baseline["gpu"]["uuid"]
    assert result["server"]["checkpoint_id"] == quality[f"{variant}_checkpoint_id"]
    assert result["requests_per_run"] == 512 and result["repeats"] == 3
    assert result["concurrency_levels"] == [1, 4, 16, 32]
    assert len(result["runs"]) == 12
    for trial in result["runs"]:
        assert trial["summary"]["failed_requests"] == 0
        print("Concurrency:", trial["concurrency"], "repeat:", trial["repeat"], trial["summary"])
    return True


FP16_SWEEP_OK = sweep("fp16")
print("FP16 sweep complete. PAUSE and report before the AWQ sweep.")

In [ ]:
assert FP16_SWEEP_OK, "Complete the FP16 sweep first"
AWQ_SWEEP_OK = sweep("awq")
print("Step 4 complete. PAUSE and report both sweeps before the laptop handoff.")

## Step 5 — Preserve artifacts and hand off to the RTX 5060 laptop

**The laptop step cannot run on Colab.** Save the cloud results and validated checkpoint
with the next cell before disconnecting. It copies to a new folder in your Drive; it
does not change model weights, upload credentials, or overwrite an earlier run folder.

Then copy the entire `models/mistral-7b-awq/` and `results/` folders into the local
repository. Tell the assistant where you copied them. Use the pinned GPU environment on the RTX 5060. Record device VRAM usage in a
separate terminal before starting the demo:

```bash
nvidia-smi --id=0 --query-gpu=timestamp,name,uuid,memory.used --format=csv,noheader,nounits --loop-ms=200 > results/edge_vram.csv
```

In another terminal, run:

```bash
python scripts/edge_demo.py
```

Stop the memory recorder with **Ctrl+C after the demo finishes**. This is a 200 ms
sampled device-wide peak, including the display and other processes, not an exact
model-only allocation peak. Preserve the CSV alongside `edge_demo.json`. The script's
utilization target is **not** a measured memory result. FP16 weights alone cannot fit fully in 8 GB.
Stop on local CUDA/kernel/OOM errors and share the traceback.

Do not proceed to the final report until successful laptop inference is verified.
You can generate the report locally from the copied artifacts; the final Colab cells
provide an alternative after uploading the real laptop output.

In [ ]:
import shutil

from google.colab import drive

assert FP16_SWEEP_OK and AWQ_SWEEP_OK, "Complete both cloud sweeps first"
stop_server()


drive.mount("/content/drive")
RUN_NAME = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
DEST = Path("/content/drive/MyDrive/llm-inference-benchmark") / RUN_NAME
DEST.mkdir(parents=True, exist_ok=False)
shutil.copytree(ROOT / "results", DEST / "results")
shutil.copytree(ROOT / "models/mistral-7b-awq", DEST / "models/mistral-7b-awq")
(DEST / "source_commit.txt").write_text(REPO_COMMIT + "\n")
print("Saved cloud artifacts:", DEST)
print("STOP HERE: copy artifacts to the laptop and complete Step 5 locally.")

### Optional: return the real laptop results to this notebook

Skip these final Colab cells if completing the report locally. Otherwise, after the
laptop run succeeds, upload **both `edge_demo.json` and `edge_vram.csv`** below. The
cell checks checkpoint identity, RTX 5060 hardware, nonempty generation, and that the
memory recorder used the same physical GPU. It reports the peak from actual samples;
this includes other processes and can miss spikes between samples.

In [ ]:
import csv
import io

from google.colab import files

assert FP16_SWEEP_OK and AWQ_SWEEP_OK
EDGE_OK = False
uploaded = files.upload()
assert {"edge_demo.json", "edge_vram.csv"} <= uploaded.keys(), (
    "Upload both real laptop files: edge_demo.json and edge_vram.csv"
)
edge = json.loads(uploaded["edge_demo.json"].decode())
assert edge["identity"] == expected_identity
assert edge["checkpoint_id"] == quality["awq_checkpoint_id"]
assert "5060" in edge["gpu"]["name"], "STOP: wrong laptop GPU"
assert edge["output_tokens"] > 0 and edge["text"].strip()
reader = csv.DictReader(
    io.StringIO(uploaded["edge_vram.csv"].decode()),
    fieldnames=("timestamp", "name", "uuid", "memory_used_mib"),
)
samples = [{key.strip(): value.strip() for key, value in row.items()} for row in reader]
assert samples, "STOP: no real VRAM samples"
gpu_uuid = edge["gpu"]["uuid"].removeprefix("GPU-").lower()
assert all(row["uuid"].removeprefix("GPU-").lower() == gpu_uuid for row in samples), (
    "STOP: VRAM GPU mismatch"
)
used_mib = [float(row["memory_used_mib"]) for row in samples]
assert all(math.isfinite(value) and value > 0 for value in used_mib)
for name in ("edge_demo.json", "edge_vram.csv"):
    (ROOT / "results" / name).write_bytes(uploaded[name])
edge_memory = {
    "source": "edge_vram.csv",
    "gpu_uuid": edge["gpu"]["uuid"],
    "sample_count": len(samples),
    "sample_interval_ms": 200,
    "sampled_device_used_peak_mib": max(used_mib),
    "notes": "Sampled device-wide peak includes other processes and may miss short spikes.",
}
(ROOT / "results/edge_vram_summary.json").write_text(json.dumps(edge_memory, indent=2) + "\n")
EDGE_OK = True
print("Laptop generation:", edge["text"])
print("Actual sampled device VRAM peak (MiB):", edge_memory["sampled_device_used_peak_mib"])
print("Step 5 verified. PAUSE and report before Step 6.")

## Step 6 — Generate the measured report

Run only after the real laptop inference and VRAM observation have been reviewed.
The plotter requires all measured cloud artifacts and checks their compatibility.
No fake/example charts are substituted on failure. It produces three PNGs, a CSV, and
`results/REPORT.md`. After review, update the README table from these files and commit
measured results in a separate, descriptive commit. Preserve failed-run logs and note
any remaining limitations. Download/save the final results before leaving Colab.

In [ ]:
from IPython.display import Image, Markdown, display

assert EDGE_OK, "Complete and review the real RTX 5060 laptop demo first"

run("scripts/plot_results.py")
display(Markdown((ROOT / "results/REPORT.md").read_text()))
for name in ("throughput.png", "latency.png", "memory_quality.png"):
    display(Image(filename=str(ROOT / "results" / name)))
shutil.copytree(ROOT / "results", DEST / "results-final")
print("Final artifacts saved:", DEST / "results-final")
print("Share the report, fill the README from measured artifacts, then commit results locally.")

## If anything fails

1. Stop; do not continue to the next stage or change packages.
2. Copy the complete error output. Command logs are under `results/`; server startup
   logs include the actual traceback and earlier attempts are archived with timestamps.
3. Share the stage, traceback, and elapsed time. Timeouts are failures, not measurements.
4. Preserve model/results directories. A new runtime may require restoring them from Drive;
   do not combine baseline/serving runs from different physical GPUs.

Known fixes included: same-interpreter vLLM launch (no reliance on notebook PATH), and
`MPLBACKEND=Agg` for every GPU subprocess. These are launch fixes; model/precision and
all pinned dependency versions remain unchanged. CPU tests validate orchestration;
GPU execution remains a manual integration check.

For incomplete HTTP streams, `scripts/diagnose_streams.py` replays selected frozen
request IDs and saves full raw SSE events separately. This is diagnosis, not a
replacement benchmark. Preserve the original failed sweep and inspect the missing
signals before changing validation or restarting a full sweep.
